In [ ]:
import numpy, pandas
import os
from skimage.io import imread, imsave
import skimage

In [ ]:
mask_dir = "2089/2089.mMACseg/"

cell_mask_file = os.path.join(mask_dir, 'masks.tif')
image_file = os.path.join(mask_dir, '../2089.mMAC.ome.tif')
imageDataChns=[0,1,2,3,4]
imageDataChnsNames = ['CK','CD11c','F480', 'CD163', 'DAPI']
proteins = ['CK','CD11c','F480', 'CD163']

protein_file = os.path.join(mask_dir, "protein.tsv")
protein_norm_file = os.path.join(mask_dir, "protein_normalized.tsv")
coordinate_file = os.path.join(mask_dir, "coordinate.tsv")

In [ ]:
cell_mask = imread(cell_mask_file)
cell_mask = cell_mask.astype(numpy.uint32)

In [ ]:
original_image = imread(image_file)

In [ ]:
original_image.shape, cell_mask.shape

In [ ]:
if original_image.shape[:2] != cell_mask.shape:
    print ("switch axis")
    original_image = numpy.transpose(original_image,[1,2,0])

In [ ]:
props = skimage.measure.regionprops_table(cell_mask, intensity_image= original_image, properties=['label', 'centroid', 'area', 'eccentricity', 'perimeter'])
cell_dataframe = pandas.DataFrame(props)
cell_dataframe = cell_dataframe.set_index('label')
cell_dataframe.index.name = 'cell'
cell_dataframe

In [ ]:
cells = cell_dataframe.index
len(cells)

In [ ]:
def proteinInCellsum(original_image, mask, coord, cell_id):
    distance = 100
    row = int(coord['centroid-0'][cell_id])
    col = int(coord['centroid-1'][cell_id])
    new_cell_id = mask[row, col]
    if new_cell_id!=0:
        row_min = max(row - distance,0)
        row_max = min(row + distance, original_image.shape[0])
        col_min = max(col - distance,0)
        col_max = min(col + distance, original_image.shape[1])
        image = original_image[row_min:row_max, col_min:col_max, :]
        values = image[ mask[row_min:row_max, col_min:col_max] == new_cell_id].sum(axis=0)[imageDataChns]
    else:
        values = [0] * len(imageDataChnsNames)
    return values

In [ ]:
def proteinInCellmean(original_image, mask, coord, cell_id):
    distance = 100
    row = int(coord['centroid-0'][cell_id])
    col = int(coord['centroid-1'][cell_id])
    new_cell_id = mask[row, col]
    if new_cell_id!=0:
        row_min = max(row - distance,0)
        row_max = min(row + distance, original_image.shape[0])
        col_min = max(col - distance,0)
        col_max = min(col + distance, original_image.shape[1])
        image = original_image[row_min:row_max, col_min:col_max, :]
        values = image[ mask[row_min:row_max, col_min:col_max] == new_cell_id].mean(axis=0)[imageDataChns]
    else:
        values = [0] * len(imageDataChnsNames)
    return values

In [ ]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from functools import partial
import multiprocessing

# Specify the desired number of CPUs
num_cpus =  multiprocessing.cpu_count()
seg = 2000

# Use ThreadPoolExecutor for parallel processing with a specified number of CPUs
with ProcessPoolExecutor(max_workers=num_cpus) as executor:
    # Define a function to process a single element
    def process_list(start):
        cells_index = list(range(start, min(start +seg, len(cells))))
        selected_cells = [cells[x] for x in cells_index]
        result = []
        for cell_id in selected_cells:
            cellProteinResult = proteinInCellmean(original_image, cell_mask, cell_dataframe, cell_id)
            result.append(cellProteinResult)
        return list(zip(selected_cells, result))
        
    # Use executor.map to apply the function to each element in parallel
    result_array = list(executor.map(process_list, list(range(0, len(cells), seg))))
    result_array = sum(result_array,[])

In [ ]:
df = pandas.DataFrame(result_array)
protein_matrix = pandas.DataFrame(df[1].tolist(), index= df[0], columns= imageDataChnsNames)
protein_matrix.index.name = 'cell'
protein_matrix

In [ ]:
intensity_toal = protein_matrix.sum(axis=1)
intensity_toal

In [ ]:
protein_matrix = protein_matrix[proteins]
protein_matrix

In [ ]:
protein_matrix.to_csv(protein_file, sep='\t')

In [ ]:
cell_dataframe.to_csv(coordinate_file, sep='\t')

# normalized protein expression by total cell intensity - step 1

In [ ]:
protein_norm_matrix_bycell = (protein_matrix.T/intensity_toal).T
protein_norm_matrix_bycell

# normalized by expression rank within each protein - step 2

In [ ]:
protein_norm_matrix = pandas.DataFrame()
protein_norm_matrix['CK_norm'] = protein_norm_matrix_bycell['CK']/protein_norm_matrix_bycell['CK'].quantile(.99)
protein_norm_matrix['F480_norm'] = protein_norm_matrix_bycell['F480']/protein_norm_matrix_bycell['F480'].quantile(.99)
protein_norm_matrix['CD163_norm'] = protein_norm_matrix_bycell['CD163']/protein_norm_matrix_bycell['CD163'].quantile(.99)
protein_norm_matrix = protein_norm_matrix.clip(upper=1)
protein_norm_matrix

In [ ]:
protein_norm_matrix.to_csv(protein_norm_file, sep='\t')